In [1]:
from pathlib import Path

import polars as pl

data_dir = Path.cwd().parent.parent.parent / "data"

raw_csv_path = data_dir / "bbbenji" / "bbbenji.csv"

df = pl.read_csv(raw_csv_path)

In [2]:
df.head()

sentence,url,fileName,commonVoiceFileName,speechPatterns,userId,sentenceId,date,numberOfClipsWithSentence,numberOfSkipsForSentence,approximateDuration
str,str,str,str,str,str,str,str,i64,i64,f64
"""The steady drip is worse than …","""https://ssr-data.s3.amazonaws.…","""64113e55bf594264a9944162.wav""","""common_voice_en_628542.mp3""","""soundRepetition""","""640c18e27484ddffc7574b87""","""6410c06a733f4c0ce3558396""","""Tue Mar 14 2023 23:41:10 GMT-0…",2,0,5.681
"""Benjamin climbed over the wall…","""https://ssr-data.s3.amazonaws.…","""64113e55bf594264a9944163.wav""","""common_voice_en_20836598.mp3""","""soundRepetition""","""640c18e27484ddffc7574b87""","""6410c068733f4c0ce35557a4""","""Tue Mar 14 2023 23:41:10 GMT-0…",2,0,9.154
"""The manufacturer of the discs …","""https://ssr-data.s3.amazonaws.…","""64113e55bf594264a9944164.wav""","""common_voice_en_26944671.mp3""","""soundRepetition""","""640c18e27484ddffc7574b87""","""6410c073733f4c0ce356374a""","""Tue Mar 14 2023 23:41:10 GMT-0…",2,0,8.43
"""The riding is located in the s…","""https://ssr-data.s3.amazonaws.…","""64113e55bf594264a9944165.wav""","""common_voice_en_19987509.mp3""","""soundRepetition""","""640c18e27484ddffc7574b87""","""6410c06a733f4c0ce3557568""","""Tue Mar 14 2023 23:41:10 GMT-0…",2,0,10.092
"""There was an accident at work,…","""https://ssr-data.s3.amazonaws.…","""64113e55bf594264a9944161.wav""","""common_voice_en_315251.mp3""","""soundRepetition""","""640c18e27484ddffc7574b87""","""6410c070733f4c0ce355eda9""","""Tue Mar 14 2023 23:41:10 GMT-0…",2,0,12.901


In [3]:
# save as parquet

parquet_path = data_dir / "bbbenji" / "bbbenji.parquet"

df.write_parquet(parquet_path)


In [4]:
# Check the schema to see available columns
df.schema


Schema([('sentence', String),
        ('url', String),
        ('fileName', String),
        ('commonVoiceFileName', String),
        ('speechPatterns', String),
        ('userId', String),
        ('sentenceId', String),
        ('date', String),
        ('numberOfClipsWithSentence', Int64),
        ('numberOfSkipsForSentence', Int64),
        ('approximateDuration', Float64)])

In [7]:
# Load the parquet file and prepare for async downloads
import asyncio
from pathlib import Path
from urllib.parse import urlparse

import aiofiles
import aiohttp

# Read the parquet file
df = pl.read_parquet(data_dir / "bbbenji" / "bbbenji.parquet")

# Set up the audio directory
audio_dir = Path(data_dir / "bbbenji" / "audio")
audio_dir.mkdir(parents=True, exist_ok=True)

# Extract URLs using Polars' built-in features
urls = df["url"].to_list()

print(f"Found {len(urls)} URLs to download")


Found 2334 URLs to download


In [8]:
async def download_file(session: aiohttp.ClientSession, url: str, save_path: Path) -> bool:
    """Download a file from URL and save it to the specified path"""
    try:
        async with session.get(url) as response:
            if response.status == 200:
                async with aiofiles.open(save_path, "wb") as f:
                    await f.write(await response.read())
                return True
            else:
                print(f"Failed to download {url}: Status {response.status}")
                return False
    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return False


async def download_all_files(urls: list[str], output_dir: Path, max_concurrent: int = 10):
    """Download all files asynchronously with concurrency control"""
    from tqdm.asyncio import tqdm as async_tqdm

    async with aiohttp.ClientSession() as session:
        semaphore = asyncio.Semaphore(max_concurrent)
        pbar = async_tqdm(total=len(urls), desc="Downloading files")

        async def download_with_semaphore(url: str):
            async with semaphore:
                # Extract filename from URL
                parsed_url = urlparse(url)
                filename = Path(parsed_url.path).name

                # If no filename in URL, generate one from the URL hash
                if not filename or filename == "/":
                    import hashlib

                    filename = hashlib.md5(url.encode()).hexdigest() + ".mp3"

                save_path = output_dir / filename

                # Skip if file already exists
                if save_path.exists():
                    pbar.set_postfix_str(f"Skipped: {filename[:30]}...")
                    pbar.update(1)
                    return True

                success = await download_file(session, url, save_path)
                if success:
                    pbar.set_postfix_str(f"Downloaded: {filename[:30]}...")
                else:
                    pbar.set_postfix_str(f"Failed: {filename[:30]}...")
                pbar.update(1)
                return success

        # Create tasks for all downloads
        tasks = [download_with_semaphore(url) for url in urls]

        # Execute all downloads concurrently
        results = await asyncio.gather(*tasks, return_exceptions=True)

        pbar.close()

        successful = sum(1 for r in results if r is True)
        failed = len(results) - successful

        print(f"\nDownload complete: {successful} successful, {failed} failed")
        return results


In [9]:
# Run the async downloads
# Use await directly since Jupyter notebooks support top-level await
results = await download_all_files(urls, audio_dir, max_concurrent=10)



Download complete: 2334 successful, 0 failed


In [ ]:
# adjust file paths to be relative to the current working directory
audio_dir_relative = "data/bbbenji/audio/"
df = df.with_columns(
    pl.concat_str([pl.lit(audio_dir_relative), pl.col("url").str.split("/").list.last()]).alias(
        "audio_path"
    )
)

# save the dataframe to a parquet file
df.write_parquet(data_dir / "bbbenji" / "bbbenji.parquet")

# read the parquet file
df = pl.read_parquet(data_dir / "bbbenji" / "bbbenji.parquet")
df.head()


sentence,url,fileName,commonVoiceFileName,speechPatterns,userId,sentenceId,date,numberOfClipsWithSentence,numberOfSkipsForSentence,approximateDuration,audio_path
str,str,str,str,str,str,str,str,i64,i64,f64,str
"""The steady drip is worse than …","""https://ssr-data.s3.amazonaws.…","""64113e55bf594264a9944162.wav""","""common_voice_en_628542.mp3""","""soundRepetition""","""640c18e27484ddffc7574b87""","""6410c06a733f4c0ce3558396""","""Tue Mar 14 2023 23:41:10 GMT-0…",2,0,5.681,"""data/bbbenji/audio/64113e55bf5…"
"""Benjamin climbed over the wall…","""https://ssr-data.s3.amazonaws.…","""64113e55bf594264a9944163.wav""","""common_voice_en_20836598.mp3""","""soundRepetition""","""640c18e27484ddffc7574b87""","""6410c068733f4c0ce35557a4""","""Tue Mar 14 2023 23:41:10 GMT-0…",2,0,9.154,"""data/bbbenji/audio/64113e55bf5…"
"""The manufacturer of the discs …","""https://ssr-data.s3.amazonaws.…","""64113e55bf594264a9944164.wav""","""common_voice_en_26944671.mp3""","""soundRepetition""","""640c18e27484ddffc7574b87""","""6410c073733f4c0ce356374a""","""Tue Mar 14 2023 23:41:10 GMT-0…",2,0,8.43,"""data/bbbenji/audio/64113e55bf5…"
"""The riding is located in the s…","""https://ssr-data.s3.amazonaws.…","""64113e55bf594264a9944165.wav""","""common_voice_en_19987509.mp3""","""soundRepetition""","""640c18e27484ddffc7574b87""","""6410c06a733f4c0ce3557568""","""Tue Mar 14 2023 23:41:10 GMT-0…",2,0,10.092,"""data/bbbenji/audio/64113e55bf5…"
"""There was an accident at work,…","""https://ssr-data.s3.amazonaws.…","""64113e55bf594264a9944161.wav""","""common_voice_en_315251.mp3""","""soundRepetition""","""640c18e27484ddffc7574b87""","""6410c070733f4c0ce355eda9""","""Tue Mar 14 2023 23:41:10 GMT-0…",2,0,12.901,"""data/bbbenji/audio/64113e55bf5…"
